In [1]:
# Stage 5 — PPE Representation Study
# 5.4 — Spatiotemporal Embedding A* (V_delta 15D)
#
# Goal: Retest A' with V_delta expanded from 4D to 15D, per Dan's suggestion.
# Same delta matrix as 5.1 (per-plant, 3160 bins × 52 weeks, norm from PPE).
# PCA to 15 components instead of 4 → V_delta_15.
# Feature vector: [Vf (15D), Vp (15D), N (1D), V_delta_15 (15D)] = 46D
# Compare against A' (35D, 4D V_delta) and A3 (32D, scalar delta).

import numpy as np
import pandas as pd
import glob
import pickle
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/scratch/ariana.l/Stage 4 Link Prediction Model/"
STAGE5_BASE = "/scratch/ariana.l/Stage 5 PPE Representation Study/"
PPE_DIR = "/scratch/ariana.l/ppe-outputs/opportunity_surface/"

In [2]:
# Cell 2 — Load existence matrices and reconstruct common bins

print("Loading existence matrices...")
F = pd.read_csv(BASE + "stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv(BASE + "stage4_P_existence_gbif_combined.csv", index_col=0)

common_bins = sorted(set(F.columns) & set(P.columns))
bin_to_idx = {b: i for i, b in enumerate(common_bins)}
n_bins = len(common_bins)  # 3160
n_weeks = 52
print(f"Common bins: {n_bins}")

def snap(x):
    return round(round(x * 2) / 2, 1)

def fmt(x):
    return f"{x:.1f}"

Loading existence matrices...
Common bins: 3160


In [3]:
# Cell 3 — Read PPE opportunity surface and collect species data
# Identical to 5.1 and 5.2

species_data = defaultdict(list)

files = sorted(glob.glob(PPE_DIR + "part_*.parquet"))
print(f"Found {len(files)} parquet files")
print("Reading opportunity surface files...")

for i, fpath in enumerate(files):
    df = pd.read_parquet(fpath, columns=['species', 'centroid_lat', 'centroid_lon', 'week', 'norm'])
    df['bin_key'] = df['centroid_lat'].map(snap).map(fmt) + '_' + df['centroid_lon'].map(snap).map(fmt)
    df = df[df['bin_key'].isin(bin_to_idx)]
    for row in df.itertuples(index=False):
        species_data[row.species].append((bin_to_idx[row.bin_key], int(row.week) - 1, row.norm))
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(files)} files, {len(species_data)} species seen so far")

print(f"\nDone. Species with PPE data: {len(species_data)}")

Found 6697 parquet files
Reading opportunity surface files...
  500/6697 files, 500 species seen so far
  1000/6697 files, 1000 species seen so far
  1500/6697 files, 1500 species seen so far
  2000/6697 files, 2000 species seen so far
  2500/6697 files, 2500 species seen so far
  3000/6697 files, 3000 species seen so far
  3500/6697 files, 3500 species seen so far
  4000/6697 files, 4000 species seen so far
  4500/6697 files, 4500 species seen so far
  5000/6697 files, 5000 species seen so far
  5500/6697 files, 5500 species seen so far
  6000/6697 files, 6000 species seen so far
  6500/6697 files, 6500 species seen so far

Done. Species with PPE data: 6697


In [4]:
# Cell 4 — Assemble delta matrix and fit PCA to 15D

plant_species = sorted(species_data.keys())
n_species = len(plant_species)
print(f"Plant species: {n_species}")

print(f"Allocating delta matrix: ({n_species}, {n_bins}, {n_weeks})...")
delta_3d = np.zeros((n_species, n_bins, n_weeks), dtype=np.float32)

print("Filling delta matrix...")
for sp_i, sp in enumerate(plant_species):
    for (bin_idx, week_idx, norm) in species_data[sp]:
        delta_3d[sp_i, bin_idx, week_idx] = norm
    if (sp_i + 1) % 1000 == 0:
        print(f"  {sp_i+1}/{n_species} species filled")

print(f"Delta matrix shape: {delta_3d.shape}")
print(f"Memory usage: {delta_3d.nbytes / 1e9:.2f} GB")

delta_flat = delta_3d.reshape(n_species, n_bins * n_weeks)
print(f"Flattened shape: {delta_flat.shape}")

# Fit PCA to 15D — expanded from 4D in A'
print("Fitting PCA (randomized, 15 components)...")
pca_delta_15 = PCA(n_components=15, svd_solver='randomized', random_state=42)
V_delta_15 = pca_delta_15.fit_transform(delta_flat)
print(f"V_delta_15 shape: {V_delta_15.shape}")
print(f"Variance explained per component: {pca_delta_15.explained_variance_ratio_.round(3)}")
print(f"Total variance explained: {pca_delta_15.explained_variance_ratio_.sum():.3f}")
print(f"\nFor reference — A' used 4D V_delta, variance explained: 38.7%")

Plant species: 6697
Allocating delta matrix: (6697, 3160, 52)...
Filling delta matrix...
  1000/6697 species filled
  2000/6697 species filled
  3000/6697 species filled
  4000/6697 species filled
  5000/6697 species filled
  6000/6697 species filled
Delta matrix shape: (6697, 3160, 52)
Memory usage: 4.40 GB
Flattened shape: (6697, 164320)
Fitting PCA (randomized, 15 components)...
V_delta_15 shape: (6697, 15)
Variance explained per component: [0.201 0.109 0.052 0.025 0.019 0.016 0.011 0.01  0.009 0.007 0.006 0.005
 0.005 0.005 0.005]
Total variance explained: 0.486

For reference — A' used 4D V_delta, variance explained: 38.7%


In [5]:
# Cell 5 — Save V_delta_15 and free memory

Vd15_df = pd.DataFrame(V_delta_15, index=plant_species, columns=[f'PC{i+1}' for i in range(15)])
Vd15_df.to_csv(STAGE5_BASE + "stage5_Vdelta_15d.csv")
print(f"Saved V_delta_15: {Vd15_df.shape}")

with open(STAGE5_BASE + "stage5_pca_delta_15d.pkl", "wb") as f:
    pickle.dump(pca_delta_15, f)
print("Saved PCA object")

del delta_3d, delta_flat
import gc; gc.collect()
print("Freed delta_3d and delta_flat from memory")

Saved V_delta_15: (6697, 15)
Saved PCA object
Freed delta_3d and delta_flat from memory


In [6]:
# Cell 6 — Reconstruct training pairs and assemble 46D feature vectors

print("Loading assets...")
Vf_df = pd.read_csv(BASE + "stage4_Vf_phenofield.csv", index_col=0)
Vp_df = pd.read_csv(BASE + "stage4_Vp_gbif.csv", index_col=0)
globi = pd.read_csv(BASE + "stage4_globi_conus_broad.csv")

F_common = F[common_bins]
P_common = P[common_bins]

# Positive pairs
pos_pairs = globi[['sourceTaxonName', 'targetTaxonName']].drop_duplicates()
pos_pairs.columns = ['pollinator', 'plant']
pos_pairs = pos_pairs[
    pos_pairs['plant'].isin(Vf_df.index) &
    pos_pairs['plant'].isin(F.index) &
    pos_pairs['plant'].isin(Vd15_df.index) &
    pos_pairs['pollinator'].isin(Vp_df.index) &
    pos_pairs['pollinator'].isin(P.index)
]
pos_pairs['label'] = 1
print(f"Positive pairs: {len(pos_pairs)}")

# Negative pairs
pos_set = set(zip(pos_pairs['pollinator'], pos_pairs['plant']))
all_plants = list(set(Vf_df.index) & set(F.index) & set(Vd15_df.index))
all_pols = list(set(Vp_df.index) & set(P.index))

np.random.seed(42)
neg_pairs = []
while len(neg_pairs) < len(pos_pairs) * 3:
    pol = np.random.choice(all_pols)
    plant = np.random.choice(all_plants)
    if (pol, plant) not in pos_set:
        neg_pairs.append((pol, plant, 0))

neg_pairs = pd.DataFrame(neg_pairs, columns=['pollinator', 'plant', 'label'])
print(f"Negative pairs: {len(neg_pairs)}")

pairs = pd.concat([pos_pairs, neg_pairs], ignore_index=True)

# Assemble 46D feature vectors
def build_features(row):
    vf  = Vf_df.loc[row.plant].values          # 15D — binary plant embedding
    vp  = Vp_df.loc[row.pollinator].values     # 15D — binary pollinator embedding
    vd  = Vd15_df.loc[row.plant].values        # 15D — spatiotemporal plant embedding
    N   = float(np.dot(F_common.loc[row.plant].values, P_common.loc[row.pollinator].values))
    return np.concatenate([vf, vp, [N], vd])   # 46D

print("Assembling feature matrix...")
X = np.vstack([build_features(row) for row in pairs.itertuples()])
y = pairs['label'].values
print(f"X shape: {X.shape}, positive rate: {y.mean():.3f}")

Loading assets...
Positive pairs: 3074
Negative pairs: 9222
Assembling feature matrix...
X shape: (12296, 46), positive rate: 0.250


In [7]:
# Cell 7 — Train/test split and logistic regression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

clf_astar = LogisticRegression(max_iter=1000, random_state=42)
clf_astar.fit(X_train, y_train)

y_prob = clf_astar.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)

print(f"\nA* (46D, V_delta 15D) Results:")
print(f"  ROC-AUC: {roc:.3f}")
print(f"  PR-AUC:  {pr:.3f}")
print(f"\nFor reference:")
print(f"  B' (31D, PMp):              ROC-AUC 0.924, PR-AUC 0.828")
print(f"  A2 (31D, binary):           ROC-AUC 0.931, PR-AUC 0.842")
print(f"  A' (35D, V_delta 4D):       ROC-AUC 0.937, PR-AUC 0.855")
print(f"  B  (31D, PMf):              ROC-AUC 0.938, PR-AUC 0.856")
print(f"  A3 (32D, scalar delta):     ROC-AUC 0.950, PR-AUC 0.868")

Train: (9836, 46), Test: (2460, 46)

A* (46D, V_delta 15D) Results:
  ROC-AUC: 0.948
  PR-AUC:  0.878

For reference:
  B' (31D, PMp):              ROC-AUC 0.924, PR-AUC 0.828
  A2 (31D, binary):           ROC-AUC 0.931, PR-AUC 0.842
  A' (35D, V_delta 4D):       ROC-AUC 0.937, PR-AUC 0.855
  B  (31D, PMf):              ROC-AUC 0.938, PR-AUC 0.856
  A3 (32D, scalar delta):     ROC-AUC 0.950, PR-AUC 0.868


In [8]:
# Cell 8 — Save model

with open(STAGE5_BASE + "stage5_Astar_logistic.pkl", "wb") as f:
    pickle.dump(clf_astar, f)
print("Saved clf_astar")

Saved clf_astar


In [10]:
# Cell 9 — Full Stage 5 results table

ap = "'"
print("=" * 65)
print(f"{'Stage 5 — PPE Representation Study: Full Results':^65}")
print("=" * 65)
print(f"{'Model':<10} {'Features':<30} {'ROC-AUC':>8} {'PR-AUC':>8}")
print("-" * 65)
print(f"{'B'+ap:<10} {'binary Vf + PMp + N (31D)':<30} {'0.924':>8} {'0.828':>8}")
print(f"{'A2':<10} {'binary Vf + Vp + N (31D)':<30} {'0.931':>8} {'0.842':>8}")
print(f"{'A'+ap:<10} {'Vf + Vp + N + V_delta 4D (35D)':<30} {'0.937':>8} {'0.855':>8}")
print(f"{'B':<10} {'PMf + Vp + N (31D)':<30} {'0.938':>8} {'0.856':>8}")
print(f"{'A*':<10} {'Vf + Vp + N + V_delta 15D (46D)':<30} {'0.948':>8} {'0.878':>8}")
print(f"{'A3':<10} {'Vf + Vp + N + delta scalar (32D)':<30} {'0.950':>8} {'0.868':>8}")
print("=" * 65)
print(f"\nBest ROC-AUC: A3  (0.950)")
print(f"Best PR-AUC:  A*  (0.878)")

        Stage 5 — PPE Representation Study: Full Results         
Model      Features                        ROC-AUC   PR-AUC
-----------------------------------------------------------------
B'         binary Vf + PMp + N (31D)         0.924    0.828
A2         binary Vf + Vp + N (31D)          0.931    0.842
A'         Vf + Vp + N + V_delta 4D (35D)    0.937    0.855
B          PMf + Vp + N (31D)                0.938    0.856
A*         Vf + Vp + N + V_delta 15D (46D)    0.948    0.878
A3         Vf + Vp + N + delta scalar (32D)    0.950    0.868

Best ROC-AUC: A3  (0.950)
Best PR-AUC:  A*  (0.878)


In [11]:
# Cell 9 — Full Stage 5 results table (reordered)

ap = "'"
print("=" * 65)
print(f"{'Stage 5 — PPE Representation Study: Full Results':^65}")
print("=" * 65)
print(f"{'Model':<10} {'Features':<30} {'ROC-AUC':>8} {'PR-AUC':>8}")
print("-" * 65)
print(f"{'A2':<10} {'binary Vf + Vp + N (31D)':<30} {'0.931':>8} {'0.842':>8}")
print(f"{'A3':<10} {'Vf + Vp + N + delta scalar (32D)':<30} {'**0.950**':>8} {'0.868':>8}")
print(f"{'A'+ap:<10} {'Vf + Vp + N + V_delta 4D (35D)':<30} {'0.937':>8} {'0.855':>8}")
print(f"{'A*':<10} {'Vf + Vp + N + V_delta 15D (46D)':<30} {'0.948':>8} {'**0.878**':>8}")
print(f"{'B':<10} {'PMf + Vp + N (31D)':<30} {'0.938':>8} {'0.856':>8}")
print(f"{'B'+ap:<10} {'binary Vf + PMp + N (31D)':<30} {'0.924':>8} {'0.828':>8}")
print("=" * 65)
print(f"\nBest ROC-AUC: A3  (0.950)")
print(f"Best PR-AUC:  A*  (0.878)")

        Stage 5 — PPE Representation Study: Full Results         
Model      Features                        ROC-AUC   PR-AUC
-----------------------------------------------------------------
A2         binary Vf + Vp + N (31D)          0.931    0.842
A3         Vf + Vp + N + delta scalar (32D) **0.950**    0.868
A'         Vf + Vp + N + V_delta 4D (35D)    0.937    0.855
A*         Vf + Vp + N + V_delta 15D (46D)    0.948 **0.878**
B          PMf + Vp + N (31D)                0.938    0.856
B'         binary Vf + PMp + N (31D)         0.924    0.828

Best ROC-AUC: A3  (0.950)
Best PR-AUC:  A*  (0.878)


In [12]:
from IPython.display import display, Markdown

table = """
| Model | Features | ROC-AUC | PR-AUC |
|-------|----------|---------|--------|
| A2  | binary Vf + Vp + N (31D)         | 0.931 | 0.842 |
| A3  | Vf + Vp + N + Δ scalar (32D)     | **0.950** | 0.868 |
| A'  | Vf + Vp + N + V_δ 4D (35D)      | 0.937 | 0.855 |
| A*  | Vf + Vp + N + V_δ 15D (46D)     | 0.948 | **0.878** |
| B   | PMf + Vp + N (31D)               | 0.938 | 0.856 |
| B'  | binary Vf + PMp + N (31D)        | 0.924 | 0.828 |
"""

display(Markdown(table))


| Model | Features | ROC-AUC | PR-AUC |
|-------|----------|---------|--------|
| A2  | binary Vf + Vp + N (31D)         | 0.931 | 0.842 |
| A3  | Vf + Vp + N + Δ scalar (32D)     | **0.950** | 0.868 |
| A'  | Vf + Vp + N + V_δ 4D (35D)      | 0.937 | 0.855 |
| A*  | Vf + Vp + N + V_δ 15D (46D)     | 0.948 | **0.878** |
| B   | PMf + Vp + N (31D)               | 0.938 | 0.856 |
| B'  | binary Vf + PMp + N (31D)        | 0.924 | 0.828 |
